In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("API_KEY")
if not api_key:
    raise RuntimeError(
        "Переменная окружения API_KEY не задана"
    )

In [61]:
# from openai import OpenAI
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',  
)

model='llama3.1:8b'

In [68]:
import json
import requests

url = "http://localhost:11434/api/show"
payload = {"name": "llama3.1:8b"}

response = requests.post(url, json=payload)
if response.status_code == 200:
    config_data = response.json()
    # Красивый вывод JSON со всеми параметрами модели
    print(json.dumps(config_data, indent=2, ensure_ascii=False))
else:
    print(f"Ошибка запроса: {response.status_code}")


{
  "license": "LLAMA 3.1 COMMUNITY LICENSE AGREEMENT\nLlama 3.1 Version Release Date: July 23, 2024\n\n“Agreement” means the terms and conditions for use, reproduction, distribution and modification of the\nLlama Materials set forth herein.\n\n“Documentation” means the specifications, manuals and documentation accompanying Llama 3.1\ndistributed by Meta at https://llama.meta.com/doc/overview.\n\n“Licensee” or “you” means you, or your employer or any other person or entity (if you are entering into\nthis Agreement on such person or entity’s behalf), of the age required under applicable laws, rules or\nregulations to provide legal consent and that has legal authority to bind your employer or such other\nperson or entity if you are entering in this Agreement on their behalf.\n\n“Llama 3.1” means the foundational large language models and software and algorithms, including\nmachine-learning model code, trained model weights, inference-enabling code, training-enabling code,\nfine-tuning en

In [96]:
resp_dict = response.json()['model_info']
print(resp_dict)
# filtered_keys = [key for key, value in resp_dict.items() if 'llama' in key or 'llama' in str(value)]
# print(filtered_keys)

sub = 'kv'

# for key, value in resp_dict.items():
#     if substring in str(value):
#         print(f"Ключ: {key} -> Значение: {value}")

filtered_dict = {k: v for k, v in resp_dict.items() if sub in str(v) or sub in k}
print(filtered_dict)

{'general.architecture': 'llama', 'general.basename': 'Meta-Llama-3.1', 'general.file_type': 15, 'general.finetune': 'Instruct', 'general.languages': ['en', 'de', 'fr', 'it', 'pt', 'hi', 'es', 'th'], 'general.license': 'llama3.1', 'general.parameter_count': 8030261312, 'general.quantization_version': 2, 'general.size_label': '8B', 'general.tags': ['facebook', 'meta', 'pytorch', 'llama', 'llama-3', 'text-generation'], 'general.type': 'model', 'llama.attention.head_count': 32, 'llama.attention.head_count_kv': 8, 'llama.attention.layer_norm_rms_epsilon': 1e-05, 'llama.block_count': 32, 'llama.context_length': 131072, 'llama.embedding_length': 4096, 'llama.feed_forward_length': 14336, 'llama.rope.dimension_count': 128, 'llama.rope.freq_base': 500000, 'llama.vocab_size': 128256, 'tokenizer.ggml.bos_token_id': 128000, 'tokenizer.ggml.eos_token_id': 128009, 'tokenizer.ggml.merges': [], 'tokenizer.ggml.model': 'gpt2', 'tokenizer.ggml.pre': 'llama-bpe', 'tokenizer.ggml.token_type': [], 'tokeniz

In [4]:
EXPECTED_FIELDS = {
    "product_id",
    "description",
    "pros",
    "cons",
    "tags",
}

MIN_DESCRIPTION_LENGTH = 120 
MAX_DESCRIPTION_LENGTH = 700 

In [10]:
class ResponseValidationError(ValueError):
    """Ошибка парсинга или проверки ответа модели."""


def count_sentences(text):
    """Число предложений в тексте. Точка внутри числа концом предложения не считается."""
    return len([s for s in re.split(r"[.!?]+(?!\d)", text) if s.strip()]) 


def parse_and_validate(content: str) -> dict[str, str]:
    """Преобразовать JSON-ответ и проверить его контракт."""

    print('>>>>>\n', content)
    try:
        data = json.loads(content)
    except json.JSONDecodeError as error:
        raise ResponseValidationError(
            f"Ответ не является корректным JSON: {error.msg}"
        ) from error

    if not isinstance(data, dict):
        raise ResponseValidationError(
            "На верхнем уровне должен находиться JSON-объект"
        )

    received_fields = set(data)
    missing_fields = EXPECTED_FIELDS - received_fields
    unexpected_fields = received_fields - EXPECTED_FIELDS

    if missing_fields:
        raise ResponseValidationError(
            "Отсутствуют обязательные поля: "
            f"{sorted(missing_fields)}"
        )

    if unexpected_fields:
        raise ResponseValidationError(
            "Получены лишние поля: "
            f"{sorted(unexpected_fields)}"
        )

    description = data["description"]

    if not isinstance(description, str):
        raise ResponseValidationError("Поле description должно быть строкой")

    description = description.strip()

    if not description:
      raise ResponseValidationError("Поле description не должно быть пустым")
      
    i = count_sentences(description)
    if i < 2 or i > 5:
      raise ResponseValidationError("Description must contain 2-5 sentences")

    if len(description) < MIN_DESCRIPTION_LENGTH:
      raise ResponseValidationError(
          "Поле description содержит больше "
          f"{MAX_DESCRIPTION_LENGTH} символов"
      )

    if len(description) > MAX_DESCRIPTION_LENGTH:
      raise ResponseValidationError(
          "Поле description содержит меньше "
          f"{MAX_DESCRIPTION_LENGTH} символов"
      )

    return {
        "description": description,
    }

In [16]:
from typing import List
from typing_extensions import Annotated

from pydantic import (
    AfterValidator,
    BaseModel,
    ConfigDict,
    Field,
    field_validator,
)


class CardRequest(BaseModel):
    """Product Card"""

    model_config = ConfigDict(extra="forbid")

    product_id: str = Field(min_length=1, max_length=50)
    description: Annotated[str, Field(min_length=120, max_length=700), AfterValidator(count_sentences)]
    pros: Annotated[List[str], Field(min_length=2, max_length=5)]
    cons: Annotated[List[str], Field(min_length=1, max_length=3)]
    tags: Annotated[List[str], Field(min_length=3, max_length=8)]


    @field_validator("description")
    @classmethod
    def validate_description(cls, value: str) -> str:
      """Удалить внешние пробелы и проверить описание."""
      value = value.strip()

      if not value:
        raise ValueError(
            "Поле summary не должно быть пустым"
        )
      
      i = count_sentences(value)
      if i < 2 or i > 5:
        raise ValueError("Description must contain 2-5 sentences")
      
      return value


    @field_validator('pros')
    @classmethod
    def check_pros(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы pros не могут быть пустыми строками")
      return v


    @field_validator('cons')
    @classmethod
    def check_cons(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы cons не могут быть пустыми строками")
      return v


    @field_validator('tags')
    @classmethod
    def check_tags(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы tags не могут быть пустыми строками")
      return v

In [17]:
response_schema = CardRequest.model_json_schema()

In [18]:
response_schema

{'additionalProperties': False,
 'description': 'Product Card',
 'properties': {'product_id': {'maxLength': 50,
   'minLength': 1,
   'title': 'Product Id',
   'type': 'string'},
  'description': {'maxLength': 700,
   'minLength': 120,
   'title': 'Description',
   'type': 'string'},
  'pros': {'items': {'type': 'string'},
   'maxItems': 5,
   'minItems': 2,
   'title': 'Pros',
   'type': 'array'},
  'cons': {'items': {'type': 'string'},
   'maxItems': 3,
   'minItems': 1,
   'title': 'Cons',
   'type': 'array'},
  'tags': {'items': {'type': 'string'},
   'maxItems': 8,
   'minItems': 3,
   'title': 'Tags',
   'type': 'array'}},
 'required': ['product_id', 'description', 'pros', 'cons', 'tags'],
 'title': 'CardRequest',
 'type': 'object'}

In [107]:
async def generate_one(
    msg: dict, 
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350}
) -> str:
  response = await client.chat.completions.create(
      model=model,  
      messages=[
          {"role": "system", "content": 
            """
              Generate a product card in Russian (utf-8) by example.
              Example:
              {
                "product_id": "P-1001",
                "description": "Компактные внутриканальные наушники с Bluetooth 5.3 и защитой IPX4. Держат заряд до 24 часов и надёжно сидят при активном движении. Подойдут для спорта и повседневного использования.",
                "pros": ["Чистый звук", "Удобная посадка", "Долгое время работы"],
                "cons": ["Быстро разряжаются на морозе"],
                "tags": ["наушники", "беспроводные", "спорт", "IPX4", "Bluetooth 5.3"]
              } 
              "description": 2-5 sentences
            """
          },
          {"role": "user", "content": f"""{msg}"""},
      ],
      temperature=params['temperature'],
      top_p=params['top_p'],
      max_tokens=params['max_tokens'],
      response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "support_request",
            "strict": True,
            "schema": CardRequest.model_json_schema(),
        },
      },
      extra_body={
        "provider": {
          "require_parameters": True,
          "top_k": params['top_k'], 
          "repetition_penalty": params['repetition_penalty'],
        },
      },
  )

  content = response.choices[0].message.content
  if not content:
    raise ResponseValidationError(
        "Empty response of model"
    )
  
  return content

In [23]:
import json

file_name = './dev.jsonl'
MAX_ATTEMPTS = 3 

cards = 0
errors = 0

with open(file_name, "r", encoding="utf-8") as file:
  for line_number, line in enumerate(file, start=1):
    clean_line = line.strip()
    if not clean_line:
      continue

    cards += 1


    try:
      data_dict = json.loads(clean_line)
      messages_history = data_dict.copy()

      for attempt in range(1, MAX_ATTEMPTS + 1):
        content = generate_one(messages_history)

        try:
          data = parse_and_validate(content)
          break

        except ResponseValidationError as error:
          errors += 1

          if attempt == MAX_ATTEMPTS:
            raise ResponseValidationError(
                "Не удалось получить корректный ответ "
                f"за {MAX_ATTEMPTS} попытки"
            ) from error

          messages_history.extend(
            [
              {
                "role": "assistant",
                "content": content,
              },
              {
                "role": "user",
                "content": (
                    "Предыдущий ответ не прошёл проверку. "
                    f"Причина: {error}. "
                    "Исправь ответ и верни только JSON "
                    "с полями category, priority и summary."
                ),
              },
            ]
          ) 

    except json.JSONDecodeError:
      errors += 1
      print(f"Error reading JSON line {line_number}")

    # for testing only
    if line_number > 0:
      break

print('>>> ', cards, errors, round(100*errors/cards, 2))

>>>>>
 {
  "product_id": "P-0027",
  "description": "Пылесос Klin Eco - компактный и мощный пылесос с контейнером, идеальный для бытового использования. Обладает мощностью всасывания 220 Вт и может работать до 60 минут без перерыва. Весит всего 1,5 кг, что делает его легким и удобным в использовании.",
  "pros": ["Удобные насадки", "Работает тихо", "Легкий вес"],
  "cons": ["Контейнер вытряхивается быстро"],
  "tags": ["пылесос", "клин", "бытовая техника", "пылесос с контейнером"]
}
>>>  1 0 0.0


In [104]:
import asyncio, time

async def run_concurrent(items, concurrency, generate_one, params):
  """Отправляет запросы параллельно, не больше `concurrency` одновременно.

  generate_one — ваша асинхронная функция, принимает один товар и возвращает
  словарь с результатом. Возвращает список результатов и общее время прогона.
  """
  sem = asyncio.Semaphore(concurrency)

  async def wrapped(item):
    async with sem:
      return await generate_one(item, params)

  started = time.perf_counter()
  results = await asyncio.gather(*[wrapped(it) for it in items])
  return results, time.perf_counter() - started

In [101]:
file_name = './dev.jsonl'
with open(file_name, 'r', encoding='utf-8') as file:
  items = file.read().splitlines()

In [ ]:
concurrency = 1
params = {
  "temperature": 0.3, 
  "top_p": 0.9, 
  "top_k": 50, 
  "repetition_penalty": 1.1, 
  "max_tokens": 350,
}
its = items[:1]

results, time_total = await run_concurrent(its, concurrency, generate_one, params)

pretty_json = json.dumps(results, indent=2, ensure_ascii=False)
print('>>> Results:\n', pretty_json)
print('>>> Total time:\n', time_total)
print('>>> Time for item:\n', round(time_total/len(its), 2))

>>> Results:
 [
  "{\n  \"product_id\": \"P-0027\",\n  \"description\": \"Пылесос Klin Eco - компактный и мощный пылесос с контейнером объемом 2 л и временем работы до 60 минут. Подойдет для уборки в квартире или доме. Имеет удобные насадки и тихий режим работы.\",\n  \"pros\": [\"Удобные насадки\", \"Тихий режим работы\", \"Контейнер вытряхивается быстро\"],\n  \"cons\": [\"Щетка забивается волосами\", \"Чистить неудобно\"],\n  \"tags\": [\"пылесос\", \"клин\", \"бытовая техника\", \"пылесос с контейнером\", \"тихий режим\"]\n}"
]
>>> Total time:
 9.514510542037897
>>> Time for item:
 9.51


In [ ]:
import pandas as pd

row_data = [params | {'Total time': time_total, 'Time for item': round(time_total/len(its), 2)} ]
df = pd.DataFrame(row_data)

print(df)

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)

In [54]:
def log_response_usage(response) -> None:
    """Записать в лог модель, токены и стоимость запроса."""

    if response.usage is None:
        logger.warning(
            "Модель %s не вернула информацию об использовании",
            response.model,
        )
        return

    usage_data = response.usage.model_dump()

    logger.info(
        (
            "model=%s prompt_tokens=%d completion_tokens=%d "
            "total_tokens=%d cost=%s"
        ),
        response.model,
        response.usage.prompt_tokens,
        response.usage.completion_tokens,
        response.usage.total_tokens,
        usage_data.get("cost"),
    ) 

In [55]:
log_response_usage(response)

AttributeError: 'Response' object has no attribute 'usage'